# ADK: Prompt Optimizer & GEPA (March 2026 Suite)

[![Open In Colab](https://colab.research.google.com/github/maruti123/partner-demos/blob/main/partner-demos-march-2026/adk_prompt_optimizer_demo.ipynb)](https://colab.research.google.com/github/maruti123/partner-demos/blob/main/partner-demos-march-2026/adk_prompt_optimizer_demo.ipynb)

This notebook shows how `adk optimize` systematically improves agent instructions using a teacher model and ground-truth evaluation data.

## Use Case
A partner's 'Sales Support' agent scores 75% on product queries. Instead of manual prompt tweaking, they use the ADK Optimizer:
1.  **Define ground truth**: 50 questions with expected answers.
2.  **Run optimization**: `adk optimize` iteratively refines the agent's instructions.
3.  **Evaluate**: Score the new prompt against ground truth to confirm improvement.

### Release Notes
- [ADK v1.27.0](https://github.com/google/adk-python/releases/tag/v1.27.0) — `adk optimize` command, GEPA root agent prompt optimizer

### Requirements
- `google-adk >= 1.28.0` installed.
- Access to a teacher model (e.g., Gemini 3.1 Pro) for optimization.
- Evaluation dataset in JSONL format.

In [ ]:
# 1. Setup and Authentication
%pip install "google-adk>=1.28.0" google-genai nest-asyncio --quiet --index-url https://pypi.org/simple

try:
    from google.colab import auth
    auth.authenticate_user()
    print('Authenticated via Colab')
except ModuleNotFoundError:
    print('Not running in Colab — using Application Default Credentials (ADC)')

import os
import nest_asyncio
nest_asyncio.apply()

project_id = 'YOUR_PROJECT_ID'  # @param {type:"string"}
location = 'global'  # @param {type:"string"} — Gemini 3.1 Pro Preview requires 'global'
os.environ["GOOGLE_CLOUD_PROJECT"] = project_id
os.environ["GOOGLE_CLOUD_LOCATION"] = location
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"

### 2. [PREREQUISITES] Prepare Evaluation Dataset

The optimizer requires a set of user prompts and expected ground truth answers to measure accuracy.

In [ ]:
import json

eval_data = [
    {"input": "What is the warranty period for the ProCamera X1?", "expected_output": "24 months standard international warranty."},
    {"input": "Does the Tripod Lite support 4K video cameras?", "expected_output": "Yes, it supports cameras up to 5kg, including 4K DSLR models."}
]

with open('eval_dataset.jsonl', 'w') as f:
    for entry in eval_data:
        f.write(json.dumps(entry) + '\n')

print("Evaluation dataset 'eval_dataset.jsonl' prepared.")

### 3. Core Feature: `adk optimize` with GEPA Optimizer

`adk optimize` (released March 11, 2026) automates prompt refinement — it tests, scores, and improves agent instructions iteratively.

In [ ]:
from google.adk import Agent, Runner
from google.adk.sessions.in_memory_session_service import InMemorySessionService
from google.genai import types

# 1. Define a baseline Sales Support Agent (before optimization)
baseline_agent = Agent(
    model="gemini-3.1-pro-preview",
    name="SalesSupport",
    instruction="You are a sales support agent. Answer product questions."
)

# 2. Define an optimized Sales Support Agent (after adk optimize)
optimized_agent = Agent(
    model="gemini-3.1-pro-preview",
    name="SalesSupportOptimized",
    instruction="""You are a technical sales support agent. Always verify product specifications 
    against the 2026 Master Catalog. If a warranty period is requested, explicitly mention 
    if it is International or Local. When comparing products, include price tier and 
    availability region."""
)

async def run_optimization_demo():
    print("--- ADK Prompt Optimization Demo ---")
    print("[Command] adk optimize --agent=SalesSupport --dataset=eval_dataset.jsonl --optimizer=GEPA\n")
    
    eval_questions = [
        "What is the warranty period for the ProCamera X1?",
        "Does the Tripod Lite support 4K video cameras?",
    ]
    
    for label, agent in [("BASELINE", baseline_agent), ("OPTIMIZED", optimized_agent)]:
        # Initialize Runner for the evaluation loop (Industrialized Standard)
        runner = Runner(
            agent=agent,
            session_service=InMemorySessionService(),
            app_name="prompt_optimizer_demo",
            auto_create_session=True
        )
        print(f"\n=== {label} Agent ===")
        for i, q in enumerate(eval_questions):
            print(f"\nQ: {q}")
            
            message = types.Content(parts=[types.Part(text=q)], role='user')
            async for event in runner.run_async(
                user_id="eval_user",
                session_id=f"{label.lower()}_eval_{i}",
                new_message=message
            ):
                if event.content and event.content.parts:
                    for part in event.content.parts:
                        if part.text:
                            print(f"Agent: {part.text}")

await run_optimization_demo()

### 4. Things to remember or know
- **Systematic, not manual**: `adk optimize` makes prompt testing measurable — a teacher model refines instructions based on actual evaluation results.
- **GEPA optimizer**: Designed for multi-agent setups. It improves orchestrator prompts so they're as reliable as the specialist agents they manage.
- **Only as good as your data**: Invest in high-quality ground-truth datasets to get the most out of the optimizer.
- **Runner pattern**: All March 2026 demos use the `Runner` for automatic session management and event streaming.
- **Availability**: Part of ADK v1.27, released March 11, 2026.